# 06 - Gold: KPIs de `ecommerce_pedidos`
Lê a Silver de pedidos, calcula os KPIs de negócio da planilha e grava cada um em **Delta** (pasta `gold/` do container da squad) e no **SQL Server** (schema `squad2`).
KPIs são recalculados inteiros a cada execução (**overwrite**); `gold_hist_kpis_pedidos` guarda uma foto por execução (**append**).

| Regra | Tabela | Alerta |
|---|---|---|
| N6 | `gold_kpi_volume_pedidos_5min` | menos de `LIMITE_PEDIDOS_5MIN` pedidos em 5 min, entre 9h e 23h |
| N7 | `gold_kpi_ticket_medio_pagamento` | Pix mais de 30% abaixo da média histórica |
| N8 | `gold_kpi_taxa_cancelamento_30min` | cancelamento > 10% numa janela de 30 min |
| N9 | `gold_kpi_receita_dia` | receita do dia difere mais de `LIMITE_VARIACAO_RECEITA` do mesmo dia da semana anterior |

Decisões: receita e ticket médio **não contam pedidos cancelados**; horários em Brasília; `metodo_pagamento` agrupado em minúsculas.

In [0]:
%run ./00_setup_config

## Parâmetros

In [0]:
from datetime import timedelta
from pyspark.sql import functions as F, Window

GRUPO, FUSO_NEGOCIO = "grupo2", "America/Sao_Paulo"
BASE = f"abfss://{SQUAD}@{_sufixo}/{GRUPO}".rstrip("/")
CAMINHO_SILVER = f"{BASE}/silver/{TABELA}"

# Limiar de N6 não definido pelo negócio: valor provisório.
LIMITE_PEDIDOS_5MIN, HORA_PICO = 1, (9, 23)
QUEDA_PIX, LIMITE_CANCELAMENTO, LIMITE_VARIACAO_RECEITA = -30.0, 10.0, 30.0
HORAS_JANELA = 24  # quanto da Silver entra nas tabelas de janela (N6 e N8)

## Leitura da Silver e instante de referência
Os KPIs são "em tempo real": medidos a partir de agora. Fora de uma janela de tempo real a Silver não recebe pedidos novos e tudo daria zero;
nesse caso a referência passa a ser o último pedido da Silver, e o notebook avisa.

In [0]:
silver = spark.read.format("delta").options(**adls_options).load(CAMINHO_SILVER)
pedidos = (silver.select("id_pedido", "status_pedido", "valor_total", F.lower("metodo_pagamento").alias("metodo_pagamento"),
                         F.expr(f"convert_timezone('{FUSO_NEGOCIO}', dt_pedido)").alias("dt_local"))
                 .withColumn("cancelado", F.col("status_pedido") == "Cancelado"))

agora, ultimo = spark.range(1).select(F.expr(f"convert_timezone('{FUSO_NEGOCIO}', current_timestamp())")).first()[0], pedidos.agg(F.max("dt_local")).first()[0]
if ultimo is None:
    raise ValueError(f"Silver vazia em {CAMINHO_SILVER}: rodar Bronze e Silver antes.")
# Literal montado como texto: um datetime Python sem fuso seria convertido pelo fuso da máquina.
ntz = lambda d: F.lit(d.strftime("%Y-%m-%d %H:%M:%S.%f")).cast("timestamp_ntz")
REF = agora if (agora - ultimo).total_seconds() <= HORAS_JANELA * 3600 else ultimo
print(f"Pedidos na Silver: {pedidos.count()} | último pedido: {ultimo} | agora: {agora:%Y-%m-%d %H:%M:%S}")
print(f"Referência dos KPIs: {REF}" + ("" if REF == agora else "  [aviso] sem pedidos nas últimas horas: usando o último pedido"))

ref, recentes = ntz(REF), pedidos.filter(F.col("dt_local") > ntz(REF - timedelta(hours=HORAS_JANELA)))

## N6 - Volume de pedidos em 5 minutos (janela deslizante)
Contagem por minuto sobre uma grade contínua, para que minutos **sem pedido** apareçam com zero: é justamente a queda que o alerta precisa enxergar.
Cada linha é a janela de 5 minutos que termina naquele minuto.

In [0]:
inicio_grade = F.date_trunc("minute", ntz(REF - timedelta(hours=HORAS_JANELA)))
grade = spark.range(1).select(F.explode(F.sequence(inicio_grade, F.date_trunc("minute", ref), F.expr("INTERVAL 1 MINUTE"))).alias("minuto"))
por_minuto = recentes.groupBy(F.date_trunc("minute", "dt_local").alias("minuto")).count()
ult5 = Window.orderBy("minuto").rowsBetween(-4, 0)

kpi_volume = (grade.join(por_minuto, "minuto", "left").fillna(0, ["count"])
    .select((F.col("minuto") - F.expr("INTERVAL 4 MINUTES")).alias("inicio_janela"), (F.col("minuto") + F.expr("INTERVAL 1 MINUTE")).alias("fim_janela"),
            F.sum("count").over(ult5).alias("qtd_pedidos"), F.hour("minuto").between(HORA_PICO[0], HORA_PICO[1] - 1).alias("horario_pico"))
    .withColumn("alerta_queda", F.col("horario_pico") & (F.col("qtd_pedidos") < LIMITE_PEDIDOS_5MIN)))
display(kpi_volume.orderBy(F.desc("fim_janela")).limit(10))

## N7 - Ticket médio por método de pagamento
Ticket do **dia da referência** contra a média dos dias anteriores na Silver. Sem histórico a variação fica nula e não há alerta.

In [0]:
validos = pedidos.filter(~F.col("cancelado"))
dia_ref = F.to_date(ref)
kpi_ticket = (validos.groupBy("metodo_pagamento").agg(
        F.count(F.when(F.to_date("dt_local") == dia_ref, 1)).alias("pedidos_dia"),
        F.round(F.avg(F.when(F.to_date("dt_local") == dia_ref, F.col("valor_total"))), 2).alias("ticket_medio_dia"),
        F.round(F.avg(F.when(F.to_date("dt_local") < dia_ref, F.col("valor_total"))), 2).alias("ticket_medio_historico"))
    .withColumn("variacao_pct", F.round((F.col("ticket_medio_dia") / F.col("ticket_medio_historico") - 1) * 100, 1))
    .withColumn("alerta_pix", F.coalesce((F.col("metodo_pagamento") == "pix") & (F.col("variacao_pct") < QUEDA_PIX), F.lit(False))))
display(kpi_ticket.orderBy("metodo_pagamento"))

## N8 - Taxa de cancelamento em janelas de 30 minutos

In [0]:
kpi_cancelamento = (recentes.groupBy(F.window("dt_local", "30 minutes")).agg(F.count("*").alias("total_pedidos"), F.sum(F.col("cancelado").cast("int")).alias("cancelados"))
    .select(F.col("window.start").alias("inicio_janela"), F.col("window.end").alias("fim_janela"), "total_pedidos", "cancelados",
            F.round(F.col("cancelados") / F.col("total_pedidos") * 100, 1).alias("taxa_cancelamento_pct"))
    .withColumn("alerta_cancelamento", F.col("taxa_cancelamento_pct") > LIMITE_CANCELAMENTO))
display(kpi_cancelamento.orderBy(F.desc("inicio_janela")))

## N9 - Receita acumulada no dia vs mesmo dia da semana anterior
Para o dia da referência, ainda em andamento, a comparação é **até o mesmo horário**: o dia de hoje até as 15h contra o mesmo dia da semana anterior até as 15h.
Dias já fechados comparam o total.

In [0]:
ate_ref = F.date_format("dt_local", "HH:mm:ss") <= F.date_format(ref, "HH:mm:ss")
por_dia = validos.groupBy(F.to_date("dt_local").alias("data")).agg(
    F.round(F.sum("valor_total"), 2).alias("receita_dia"), F.count("*").alias("pedidos_dia"),
    F.round(F.sum(F.when(ate_ref, F.col("valor_total")).otherwise(0)), 2).alias("_ate_horario"))
anterior = por_dia.select(F.date_add("data", 7).alias("data"), F.col("receita_dia").alias("_ant_total"), F.col("_ate_horario").alias("_ant_ate"))

kpi_receita = (por_dia.filter(F.col("data") <= dia_ref).join(anterior, "data", "left")
    .withColumn("dia_em_andamento", F.col("data") == dia_ref)
    .withColumn("receita_comparada", F.when(F.col("dia_em_andamento"), F.col("_ate_horario")).otherwise(F.col("receita_dia")))
    .withColumn("receita_semana_anterior", F.when(F.col("dia_em_andamento"), F.col("_ant_ate")).otherwise(F.col("_ant_total")))
    .withColumn("variacao_pct", F.round((F.col("receita_comparada") / F.col("receita_semana_anterior") - 1) * 100, 1))
    .withColumn("alerta_receita", F.coalesce(F.abs("variacao_pct") > LIMITE_VARIACAO_RECEITA, F.lit(False)))
    .drop("_ate_horario", "_ant_total", "_ant_ate"))
display(kpi_receita.orderBy(F.desc("data")))

## Histórico (append)
Uma linha por KPI e por dimensão a cada execução, com o valor no instante da referência. É o que permite acompanhar a evolução e comparar médias históricas no futuro.

In [0]:
def linha(kpi, df, dimensao, valor, alerta):
    return df.select(F.lit(kpi).alias("kpi"), F.col(dimensao).cast("string").alias("dimensao"), F.col(valor).cast("double").alias("valor"), F.col(alerta).alias("alerta"))

atual = F.col("fim_janela") == F.date_trunc("minute", ref) + F.expr("INTERVAL 1 MINUTE")
hist = (linha("volume_pedidos_5min", kpi_volume.filter(atual), "fim_janela", "qtd_pedidos", "alerta_queda")
    .unionByName(linha("ticket_medio_pagamento", kpi_ticket, "metodo_pagamento", "ticket_medio_dia", "alerta_pix"))
    .unionByName(linha("taxa_cancelamento_30min", kpi_cancelamento.filter((F.col("inicio_janela") <= ref) & (ref < F.col("fim_janela"))), "inicio_janela", "taxa_cancelamento_pct", "alerta_cancelamento"))
    .unionByName(linha("receita_dia", kpi_receita.filter("dia_em_andamento"), "data", "receita_comparada", "alerta_receita"))
    .select(F.current_timestamp().alias("calculado_em"), ref.alias("referencia"), "*"))
display(hist)

## Gravação: Delta + SQL Server
Delta na pasta `gold/` do container da squad; SQL Server pelo conector nativo `sqlserver` (o JDBC genérico é bloqueado para escrita no Serverless).
Uma falha numa tabela não interrompe as outras; o resumo no fim lista o que falhou.

In [0]:
saidas = {"gold_kpi_volume_pedidos_5min_d4": (kpi_volume, "overwrite"), "gold_kpi_ticket_medio_pagamento_d4": (kpi_ticket, "overwrite"),
          "gold_kpi_taxa_cancelamento_30min_d4": (kpi_cancelamento, "overwrite"), "gold_kpi_receita_dia_d4": (kpi_receita, "overwrite"),
          "gold_hist_kpis_pedidos_d4": (hist, "append")}
falhas = []
for nome, (df, modo) in saidas.items():
    # SQL Server não tem TIMESTAMP_NTZ: vai como timestamp comum (horário de Brasília).
    df = df.select([F.col(c).cast("timestamp").alias(c) if t == "timestamp_ntz" else F.col(c) for c, t in df.dtypes])
    for destino, gravar in [("delta", lambda: df.write.format("delta").options(**adls_options).mode(modo).option("overwriteSchema", str(modo == "overwrite").lower()).save(f"{BASE}/gold/{nome}")),
                            ("sqlserver", lambda: df.write.format("sqlserver").options(**opcoes_sqlserver(f"{SQUAD}.{nome}")).mode(modo).save())]:
        try:
            gravar()
        except Exception as e:
            falhas.append((nome, destino, str(e).splitlines()[0][:200]))
    print(f"{nome:36} {modo:9} {df.count():>6} linhas")

for f in falhas:
    print("[falha]", *f)
if falhas:
    raise RuntimeError(f"{len(falhas)} gravação(ões) falharam")

## Conferência no SQL Server

In [0]:
for nome in saidas:
    sql = spark.read.format("sqlserver").options(**opcoes_sqlserver(f"{SQUAD}.{nome}")).load()
    print(f"{SQUAD}.{nome:36} {sql.count():>6} linhas no SQL Server")
alertas = hist.filter("alerta").select("kpi", "dimensao", "valor")
print(f"\nAlertas ativos na referência {REF}: {alertas.count()}")
display(alertas)

## Resumo
_Preencher após rodar no Databricks: volume por janela, ticket por método, taxa de cancelamento e receita do dia, quais alertas dispararam e o que significam para o negócio._